# Entropy Signal Analysis

Analyzes `BudgetLogitProcessor.entropy_log` to characterize the token-level
entropy signal during reasoning chain generation.

Per CLAUDE.md Section 9: entropy is logged per generation step.
This notebook investigates whether entropy drop at natural `</think>` delimiter
is a reliable early-stopping signal.

**Week 1 deliverable:** Validate entropy signal on 50 sample reasoning traces.

In [ ]:
"""
Entropy Signal Analysis — BudgetLogitProcessor
CLAUDE.md Section 9.7: entropy is telemetry ONLY; never controls stopping.

This notebook analyzes entropy traces from Week 1 baseline runs to determine
whether entropy-based adaptive early stopping is worth implementing as a
future extension.

BLOCKED until GPU node Week 1 inference runs produce entropy_log data.
The cells below are ready to execute once entropy trace data is available.
"""
print("Entropy Signal Analysis notebook — loaded.")
print("Status: BLOCKED — requires entropy trace data from GPU node Week 1 runs.")
print("Run baseline_measurements.py --baseline 1 first, then load the trace JSONs below.")

In [ ]:
import json
import pathlib
import numpy as np

# ── Load entropy traces from baseline 1 results ────────────────────────────────
# After running: python evaluation/baseline_measurements.py --baseline 1
# Results are in evaluation/results/baseline_1_*.json

RESULTS_DIR = pathlib.Path("../evaluation/results")

def load_entropy_traces(results_dir: pathlib.Path) -> list[dict]:
    """Load all entropy_log arrays from baseline 1 result files."""
    traces = []
    for path in sorted(results_dir.glob("baseline_1_*.json")):
        with open(path) as f:
            data = json.load(f)
        for result in data.get("results", []):
            entropy_log = result.get("entropy_log", [])
            if entropy_log:
                traces.append({
                    "entropy_log": entropy_log,
                    "stop_reason": result.get("stop_reason", "unknown"),
                    "reasoning_tokens": result.get("reasoning_tokens", 0),
                    "budget_class": result.get("budget_class", "unknown"),
                    "dataset": data.get("dataset", "unknown"),
                })
    return traces

traces = load_entropy_traces(RESULTS_DIR)
print(f"Loaded {len(traces)} entropy traces")
if not traces:
    print("No traces found. Run baseline_measurements.py --baseline 1 on the GPU node first.")

In [ ]:
import matplotlib.pyplot as plt

# ── Analysis 1: Entropy over token position ────────────────────────────────────
# Do natural_boundary traces show a clear entropy drop before </think>?

natural_traces = [t for t in traces if t["stop_reason"] == "natural_boundary"]
exhausted_traces = [t for t in traces if t["stop_reason"] == "budget_exhausted"]

print(f"Natural boundary traces: {len(natural_traces)}")
print(f"Budget exhausted traces: {len(exhausted_traces)}")

if natural_traces:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot normalized entropy over token position for natural completion traces
    ax = axes[0]
    for trace in natural_traces[:20]:   # Show up to 20
        n = len(trace["entropy_log"])
        positions = np.linspace(0, 1, n)
        ax.plot(positions, trace["entropy_log"], alpha=0.3, linewidth=0.8)
    ax.set_title("Entropy over token position (natural boundary)")
    ax.set_xlabel("Normalized token position (0=start, 1=</think>)")
    ax.set_ylabel("Entropy (top-20 tokens)")
    ax.set_xlim(0, 1)

    # Mean entropy by position bucket for natural vs exhausted
    ax = axes[1]
    def bucket_mean(trace_list, n_buckets=20):
        bucket_sums = np.zeros(n_buckets)
        bucket_counts = np.zeros(n_buckets)
        for t in trace_list:
            log = t["entropy_log"]
            for i, e in enumerate(log):
                bucket = min(int(i / len(log) * n_buckets), n_buckets - 1)
                bucket_sums[bucket] += e
                bucket_counts[bucket] += 1
        with np.errstate(invalid="ignore"):
            return np.where(bucket_counts > 0, bucket_sums / bucket_counts, np.nan)

    x = np.linspace(0, 1, 20)
    if natural_traces:
        ax.plot(x, bucket_mean(natural_traces), label="natural_boundary", color="green")
    if exhausted_traces:
        ax.plot(x, bucket_mean(exhausted_traces), label="budget_exhausted", color="red")
    ax.set_title("Mean entropy by position bucket")
    ax.set_xlabel("Normalized position")
    ax.set_ylabel("Mean entropy")
    ax.legend()

    plt.tight_layout()
    plt.savefig("entropy_by_position.png", dpi=150)
    plt.show()
    print("Figure saved: entropy_by_position.png")
else:
    print("No traces to plot yet.")

## Analysis 2: Entropy Drop Signal

Key question: Is there a statistically reliable entropy drop in the N tokens
before a natural `</think>` delimiter vs. a random position in the same trace?

If yes: entropy-based adaptive early stopping may be a viable future extension.
If no: entropy is confirmed as monitoring-only telemetry (current design is correct).

In [ ]:
from scipy import stats

WINDOW = 10   # Tokens before end to measure entropy

def extract_tail_entropy(trace, window=WINDOW):
    """Extract mean entropy over last `window` tokens of reasoning phase."""
    log = trace["entropy_log"]
    if len(log) < window:
        return np.mean(log)
    return np.mean(log[-window:])

def extract_mid_entropy(trace, window=WINDOW):
    """Extract mean entropy from the middle section (baseline comparison)."""
    log = trace["entropy_log"]
    mid = len(log) // 2
    return np.mean(log[max(0, mid - window//2): mid + window//2])

if natural_traces:
    tail_entropies = [extract_tail_entropy(t) for t in natural_traces]
    mid_entropies  = [extract_mid_entropy(t)  for t in natural_traces]

    t_stat, p_value = stats.ttest_rel(tail_entropies, mid_entropies)

    print(f"Tail entropy (last {WINDOW} tokens before </think>): {np.mean(tail_entropies):.4f} ± {np.std(tail_entropies):.4f}")
    print(f"Mid entropy  (window around position 50%):           {np.mean(mid_entropies):.4f} ± {np.std(mid_entropies):.4f}")
    print(f"Paired t-test: t={t_stat:.3f}, p={p_value:.4f}")

    if p_value < 0.05 and np.mean(tail_entropies) < np.mean(mid_entropies):
        print("\nCONCLUSION: Statistically significant entropy DROP before natural </think>.")
        print("  → Entropy-based adaptive stopping may be viable as a future extension.")
    else:
        print("\nCONCLUSION: No reliable entropy drop signal detected.")
        print("  → Current design (budget ceiling only) is confirmed as correct.")
        print("  → Entropy remains monitoring-only telemetry per CLAUDE.md Section 9.7.")
else:
    print("No natural boundary traces available. Blocked until GPU node data is loaded.")